# Apache Spark — Spark SQL (Complete Guide)

This notebook collects **everything about Spark SQL** from the Day 3 – Day 4 lectures and labs: running SQL on DataFrames, the SQL ↔ DataFrame equivalence, databases, managed and unmanaged tables, views, querying files directly with SQL, and SQL on streaming data.

| # | Section | What you will learn |
|---|---------|---------------------|
| | **Part 1 — Getting started** | |
| 0 | Setup | Start Spark and point it to the data folder |
| 1 | What is Spark SQL? | The engine, the Catalyst optimizer, `explain()` |
| 2 | From DataFrame to SQL | Load data with a schema and register a **temporary view** |
| | **Part 2 — Querying with SQL** | |
| 3 | Basic queries | `SELECT`, `WHERE`, `ORDER BY`, `IN` — with DataFrame equivalents |
| 4 | `CASE WHEN` | Labelling rows, a boundary bug to avoid, `when/otherwise` equivalent |
| 5 | `GROUP BY` and `HAVING` | Aggregations in SQL and in the DataFrame API |
| 6 | Exercises (with solutions) | Months with most delays, converting SQL to DataFrame code |
| | **Part 3 — Databases, tables & views** | |
| 7 | Databases | `CREATE / USE / DROP DATABASE`, the catalog |
| 8 | Managed vs unmanaged tables | Who owns the data, what `DROP TABLE` deletes |
| 9 | Views | Temporary vs global temporary views |
| 10 | Reading tables into DataFrames | `spark.sql`, `spark.table` |
| 11 | SQL directly on files | Views and tables `USING parquet / json / csv` |
| | **Part 4 — Extras** | |
| 12 | UDFs in SQL | `spark.udf.register` |
| 13 | SQL on streaming data | Temp views on streaming DataFrames |
| | **Part 5 — Wrap-up** | |
| 14 | Clean-up | Drop the database and the leftover folders |
| 15 | 📋 Cheat sheet | A summary of the whole notebook |

> The companion notebook **`Spark_DataFrame_API.ipynb`** covers the DataFrame API in depth (schemas, columns, missing data, UDFs, data sources, streaming).

---
## 0. Setup

In [1]:
import warnings
warnings.filterwarnings('ignore')

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import *

from IPython.display import display, HTML
display(HTML("<style>pre { white-space: pre !important; } </style>"))   # don't wrap wide tables

spark = (SparkSession.builder
         .appName("SparkSQL")
         .enableHiveSupport()      # keep databases & tables in the cluster's Hive metastore
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")   # hide INFO/WARN log lines so cell outputs stay readable
print("catalog:", spark.conf.get("spark.sql.catalogImplementation"))
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/30 05:27:29 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


26/10/30 05:27:29 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


catalog: hive


> **Where are databases and tables remembered?**
> - **With** `.enableHiveSupport()` (as above), Spark stores the catalog — database names, table names, schemas, locations — in the **Hive metastore** (service `thrift://hive:9083`, backed by PostgreSQL). Tables still exist after Spark stops, and Hive / Beeline see the same databases and tables. You will see your earlier Hive labs' databases in `SHOW DATABASES`.
> - **Without** it, Spark keeps the catalog **in memory**: the data files are still written to HDFS, but the table names are forgotten when the session ends.

All data lives in **HDFS**, because the executors run on the worker machines and cannot see files on this machine's local disk.

In [2]:
# No scheme -> the cluster's default filesystem, i.e. hdfs://bdss/user/user/...
DATA_DIR = "/user/user/data/spark_course"          # course input files
OUT_DIR  = "/user/user/data/spark_course_output"   # files this notebook writes
print("warehouse:", spark.conf.get("spark.sql.warehouse.dir"))

warehouse: hdfs://bdss/user/hive/warehouse


---
## 1. What is Spark SQL?

**Spark SQL** is the Spark module for **structured data** processing. Unlike plain RDDs, it knows the **structure** of the data (the schema) and of the computation, and it uses that extra information to optimize the work.

- You can talk to it with **SQL queries** or with the **DataFrame API** — both run on the **same execution engine**, so you can switch between them freely and use whichever expresses a step more naturally.
- `spark.sql("SELECT ...")` always **returns a DataFrame**, so you can continue with DataFrame operations on the result.
- Spark SQL supports **ANSI SQL:2003**–compatible queries.

### What the Spark SQL engine gives you
1. One unified abstraction (DataFrames / Datasets) across Java, Scala, Python and R.
2. Connection to the **Apache Hive metastore** and Hive tables.
3. Reading and writing structured formats (JSON, CSV, Text, Avro, Parquet, ORC, ...) and turning them into temporary tables.
4. An interactive **Spark SQL shell** (`spark-sql`) for quick exploration.
5. A bridge to external tools through standard **JDBC/ODBC** connectors.
6. A programmatic interface to tables and views stored in databases.

At its core are two components: the **Catalyst optimizer** and **Project Tungsten** (efficient memory management and code generation).

### The Catalyst optimizer *(optional reading)*
Catalyst takes your query — SQL *or* DataFrame code — and turns it into an execution plan in four phases:
1. **Analysis** — resolve table and column names using the catalog → *analyzed logical plan*.
2. **Logical optimization** — apply rules (push filters down, prune unused columns, fold constants, ...) → *optimized logical plan*.
3. **Physical planning** — generate possible physical plans and choose the cheapest one.
4. **Code generation** — generate compact Java bytecode to run on each machine.

You can see the plans with `.explain()`. We will use it in section 3 to **prove** that SQL and DataFrame code produce the same plan.

---
## 2. From DataFrame to SQL: temporary views

### 2.1 The dataset — US flight delays
| Column | Meaning |
|--------|---------|
| `date` | string like `02190925` → **Feb 19, 09:25** (format `MMddHHmm`) |
| `delay` | delay in minutes between scheduled and actual departure; **negative = left early** |
| `distance` | distance in miles from origin to destination |
| `origin` | origin airport IATA code (e.g. `SFO`) |
| `destination` | destination airport IATA code (e.g. `ORD`) |

In [3]:
flights_path = f"{DATA_DIR}/departuredelays.csv"
df_schema = "date STRING, delay INT, distance INT, origin STRING, destination STRING"

### 2.2 Reading the CSV — four variants and why the last one is right

In [4]:
# (a) No options: columns are _c0.._c4, everything is a string, and the header line is a data row
spark.read.csv(flights_path).show(2)

+--------+-----+--------+------+-----------+
|     _c0|  _c1|     _c2|   _c3|        _c4|
+--------+-----+--------+------+-----------+
|    date|delay|distance|origin|destination|
|01011245|    6|     602|   ABE|        ATL|
+--------+-----+--------+------+-----------+
only showing top 2 rows


In [5]:
# (b) inferSchema + header: good names and types, but Spark reads the WHOLE 33 MB file once just to guess types
spark.read.csv(flights_path, inferSchema=True, header=True).printSchema()

root
 |-- date: integer (nullable = true)
 |-- delay: integer (nullable = true)
 |-- distance: integer (nullable = true)
 |-- origin: string (nullable = true)
 |-- destination: string (nullable = true)



In [6]:
# (c) Schema but NO header=True: the header line is read as data -> a junk first row with nulls
spark.read.csv(flights_path, schema=df_schema).show(2)

+--------+-----+--------+------+-----------+
|    date|delay|distance|origin|destination|
+--------+-----+--------+------+-----------+
|    date| NULL|    NULL|origin|destination|
|01011245|    6|     602|   ABE|        ATL|
+--------+-----+--------+------+-----------+
only showing top 2 rows


In [7]:
# (d) Schema + header=True: correct names, correct types, header skipped, no inference job
df = spark.read.csv(flights_path, schema=df_schema, header=True)
df.printSchema()
df.show(5)

root
 |-- date: string (nullable = true)
 |-- delay: integer (nullable = true)
 |-- distance: integer (nullable = true)
 |-- origin: string (nullable = true)
 |-- destination: string (nullable = true)

+--------+-----+--------+------+-----------+
|    date|delay|distance|origin|destination|
+--------+-----+--------+------+-----------+
|01011245|    6|     602|   ABE|        ATL|
|01020600|   -8|     369|   ABE|        DTW|
|01021245|   -2|     602|   ABE|        ATL|
|01020605|   -4|     602|   ABE|        ATL|
|01031245|   -4|     602|   ABE|        ATL|
+--------+-----+--------+------+-----------+
only showing top 5 rows


### 2.3 Register the DataFrame as a temporary view
A **temporary view** gives a DataFrame a **name** that SQL queries can use, like a table.
- It holds **no data of its own** — it is just a name for the DataFrame's query plan.
- Like everything in Spark it is **lazy**: it is not kept in memory unless you `cache()` the DataFrame.
- It disappears when the SparkSession ends.

In [8]:
df.cache()     # we will query this data many times
df.createOrReplaceTempView("us_delay_flights_tbl")

---
## 3. Basic queries
These queries are no different from what you would write against MySQL or PostgreSQL.
`spark.sql(...)` returns a **DataFrame** — it does nothing until an action such as `.show()`.

### 3.1 Flights longer than 1,000 miles

In [9]:
df_long = spark.sql("""
    SELECT distance, origin, destination
    FROM us_delay_flights_tbl
    WHERE distance > 1000
""")
print(type(df_long))       # a normal DataFrame
df_long.show(5)

<class 'pyspark.sql.classic.dataframe.DataFrame'>


+--------+------+-----------+
|distance|origin|destination|
+--------+------+-----------+
|    1586|   ABQ|        JFK|
|    1586|   ABQ|        JFK|
|    1586|   ABQ|        JFK|
|    1586|   ABQ|        JFK|
|    1586|   ABQ|        JFK|
+--------+------+-----------+
only showing top 5 rows


In [10]:
# ORDER BY ... DESC: the longest flights first
spark.sql("""
    SELECT distance, origin, destination
    FROM us_delay_flights_tbl
    WHERE distance > 1000
    ORDER BY distance DESC
""").show(10)

+--------+------+-----------+
|distance|origin|destination|
+--------+------+-----------+
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
+--------+------+-----------+
only showing top 10 rows


All the longest flights are between **Honolulu (HNL)** and **New York (JFK)**.

**The same query with the DataFrame API:**

In [11]:
(df.select("distance", "origin", "destination")
   .where(F.col("distance") > 1000)
   .orderBy(F.desc("distance"))
   .show(10))

# or
(df.select("distance", "origin", "destination")
   .where("distance > 1000")
   .orderBy("distance", ascending=False)
   .show(10))

+--------+------+-----------+
|distance|origin|destination|
+--------+------+-----------+
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
+--------+------+-----------+
only showing top 10 rows


+--------+------+-----------+
|distance|origin|destination|
+--------+------+-----------+
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
|    4330|   HNL|        JFK|
+--------+------+-----------+
only showing top 10 rows


**Proof that both are the same to Spark** — compare the physical plans:

In [12]:
spark.sql("SELECT distance, origin, destination FROM us_delay_flights_tbl WHERE distance > 1000 ORDER BY distance DESC").explain()

df.select("distance", "origin", "destination").where(F.col("distance") > 1000).orderBy(F.desc("distance")).explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [distance#95 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(distance#95 DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=141]
      +- Filter (isnotnull(distance#95) AND (distance#95 > 1000))
         +- InMemoryTableScan [distance#95, origin#96, destination#97], [isnotnull(distance#95), (distance#95 > 1000)]
               +- InMemoryRelation [date#93, delay#94, distance#95, origin#96, destination#97], StorageLevel(disk, memory, deserialized, 1 replicas)
                     +- FileScan csv [date#93,delay#94,distance#95,origin#96,destination#97] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[hdfs://bdss/user/user/data/spark_course/departuredelays.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<date:string,delay:int,distance:int,origin:string,destination:string>


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [distance#95 DESC NULLS LAS

In [13]:
# explain(True) shows all the Catalyst phases: parsed -> analyzed -> optimized logical plan -> physical plan
df.select("distance", "origin").where("distance > 1000").explain(True)

== Parsed Logical Plan ==
'Filter ('distance > 1000)
+- Project [distance#95, origin#96]
   +- Relation [date#93,delay#94,distance#95,origin#96,destination#97] csv

== Analyzed Logical Plan ==
distance: int, origin: string
Filter (distance#95 > 1000)
+- Project [distance#95, origin#96]
   +- Relation [date#93,delay#94,distance#95,origin#96,destination#97] csv

== Optimized Logical Plan ==
Project [distance#95, origin#96]
+- Filter (isnotnull(distance#95) AND (distance#95 > 1000))
   +- InMemoryRelation [date#93, delay#94, distance#95, origin#96, destination#97], StorageLevel(disk, memory, deserialized, 1 replicas)
         +- FileScan csv [date#93,delay#94,distance#95,origin#96,destination#97] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[hdfs://bdss/user/user/data/spark_course/departuredelays.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<date:string,delay:int,distance:int,origin:string,destination:string>

== Physical Plan ==
*(

### 3.2 Flights from San Francisco (SFO) to Chicago (ORD) with at least a two-hour delay

In [14]:
spark.sql("""
    SELECT date, delay, origin, destination
    FROM us_delay_flights_tbl
    WHERE delay > 120 AND origin = 'SFO' AND destination = 'ORD'
    ORDER BY delay DESC
""").show(10)

+--------+-----+------+-----------+
|    date|delay|origin|destination|
+--------+-----+------+-----------+
|02190925| 1638|   SFO|        ORD|
|01031755|  396|   SFO|        ORD|
|01022330|  326|   SFO|        ORD|
|01051205|  320|   SFO|        ORD|
|01190925|  297|   SFO|        ORD|
|02171115|  296|   SFO|        ORD|
|01071040|  279|   SFO|        ORD|
|01051550|  274|   SFO|        ORD|
|03120730|  266|   SFO|        ORD|
|01261104|  258|   SFO|        ORD|
+--------+-----+------+-----------+
only showing top 10 rows


There were many seriously delayed flights between these two cities on different dates. *(Exercise 6.1 asks which months they happened in.)*

### 3.3 Both directions with `IN`

In [15]:
spark.sql("""
    SELECT *
    FROM us_delay_flights_tbl
    WHERE origin IN ('SFO', 'ORD') AND destination IN ('SFO', 'ORD') AND delay > 120
    ORDER BY delay DESC
""").show(10)

+--------+-----+--------+------+-----------+
|    date|delay|distance|origin|destination|
+--------+-----+--------+------+-----------+
|02190925| 1638|    1604|   SFO|        ORD|
|01031755|  396|    1604|   SFO|        ORD|
|01051915|  385|    1604|   ORD|        SFO|
|01081525|  372|    1604|   ORD|        SFO|
|01271820|  352|    1604|   ORD|        SFO|
|01020948|  342|    1604|   ORD|        SFO|
|01022330|  326|    1604|   SFO|        ORD|
|01051205|  320|    1604|   SFO|        ORD|
|02171205|  303|    1604|   ORD|        SFO|
|02281430|  302|    1604|   ORD|        SFO|
+--------+-----+--------+------+-----------+
only showing top 10 rows


In [16]:
# DataFrame equivalent: Column.isin(...)
(df.where(F.col('origin').isin('SFO', 'ORD') & F.col('destination').isin('SFO', 'ORD') & (F.col('delay') > 120))
   .orderBy(F.desc('delay'))
   .show(10))

+--------+-----+--------+------+-----------+
|    date|delay|distance|origin|destination|
+--------+-----+--------+------+-----------+
|02190925| 1638|    1604|   SFO|        ORD|
|01031755|  396|    1604|   SFO|        ORD|
|01051915|  385|    1604|   ORD|        SFO|
|01081525|  372|    1604|   ORD|        SFO|
|01271820|  352|    1604|   ORD|        SFO|
|01020948|  342|    1604|   ORD|        SFO|
|01022330|  326|    1604|   SFO|        ORD|
|01051205|  320|    1604|   SFO|        ORD|
|02171205|  303|    1604|   ORD|        SFO|
|02281430|  302|    1604|   ORD|        SFO|
+--------+-----+--------+------+-----------+
only showing top 10 rows


---
## 4. `CASE WHEN` — labelling rows
We want to label every flight with a human-readable delay category in a new column `Flight_Delays`:
Very Long (> 6 h), Long (2–6 h), Short (1–2 h), Tolerable (0–1 h), No delay, Early.

This is the version written in the lecture:

In [17]:
spark.sql("""
    SELECT delay, origin, destination,
           CASE
               WHEN delay > 360                 THEN 'Very Long Delays'
               WHEN delay > 120 AND delay < 360 THEN 'Long Delays'
               WHEN delay > 60  AND delay < 120 THEN 'Short Delays'
               WHEN delay > 0   AND delay < 60  THEN 'Tolerable Delays'
               WHEN delay = 0                   THEN 'No Delays'
               ELSE 'Early'
           END AS Flight_Delays
    FROM us_delay_flights_tbl
    ORDER BY origin, delay DESC
""").show(10)

+-----+------+-----------+-------------+
|delay|origin|destination|Flight_Delays|
+-----+------+-----------+-------------+
|  333|   ABE|        ATL|  Long Delays|
|  305|   ABE|        ATL|  Long Delays|
|  275|   ABE|        ATL|  Long Delays|
|  257|   ABE|        ATL|  Long Delays|
|  247|   ABE|        DTW|  Long Delays|
|  247|   ABE|        ATL|  Long Delays|
|  219|   ABE|        ORD|  Long Delays|
|  211|   ABE|        ATL|  Long Delays|
|  197|   ABE|        DTW|  Long Delays|
|  192|   ABE|        ORD|  Long Delays|
+-----+------+-----------+-------------+
only showing top 10 rows


### 4.1 Spot the bug: boundary values
What label does a flight that is **exactly 60, 120 or 360 minutes late** get?
`360` is not `> 360` and not `< 360`; `120` is not `> 120` and not `< 120` ... none of the conditions match, so it falls into `ELSE` and is labelled **'Early'**!

In [18]:
spark.sql("""
    SELECT delay,
           CASE
               WHEN delay > 360                 THEN 'Very Long Delays'
               WHEN delay > 120 AND delay < 360 THEN 'Long Delays'
               WHEN delay > 60  AND delay < 120 THEN 'Short Delays'
               WHEN delay > 0   AND delay < 60  THEN 'Tolerable Delays'
               WHEN delay = 0                   THEN 'No Delays'
               ELSE 'Early'
           END AS Flight_Delays
    FROM us_delay_flights_tbl
    WHERE delay IN (60, 120, 360)
""").groupBy('delay', 'Flight_Delays').count().orderBy('delay').show()

+-----+-------------+-----+
|delay|Flight_Delays|count|
+-----+-------------+-----+
|   60|        Early| 2000|
|  120|        Early|  536|
|  360|        Early|   20|
+-----+-------------+-----+



**The fix:** `CASE` checks the conditions **top to bottom and stops at the first match**. Order them from largest to smallest and you only need one comparison each — no gaps possible:

In [19]:
df_labeled_sql = spark.sql("""
    SELECT delay, origin, destination,
           CASE
               WHEN delay > 360 THEN 'Very Long Delays'
               WHEN delay > 120 THEN 'Long Delays'
               WHEN delay > 60  THEN 'Short Delays'
               WHEN delay > 0   THEN 'Tolerable Delays'
               WHEN delay = 0   THEN 'No Delays'
               ELSE 'Early'
           END AS Flight_Delays
    FROM us_delay_flights_tbl
""")
df_labeled_sql.where("delay IN (60, 120, 360)").groupBy('delay', 'Flight_Delays').count().orderBy('delay').show()

+-----+----------------+-----+
|delay|   Flight_Delays|count|
+-----+----------------+-----+
|   60|Tolerable Delays| 2000|
|  120|    Short Delays|  536|
|  360|     Long Delays|   20|
+-----+----------------+-----+



### 4.2 The DataFrame equivalent: `when()` / `otherwise()`
**Any SQL query can be written with the DataFrame API** — but sometimes SQL is easier to read. Pick whichever is clearer for the task.

In [20]:
df_labeled_api = df.select(
    'delay', 'origin', 'destination',
    F.when(F.col('delay') > 360, 'Very Long Delays')
     .when(F.col('delay') > 120, 'Long Delays')
     .when(F.col('delay') > 60,  'Short Delays')
     .when(F.col('delay') > 0,   'Tolerable Delays')
     .when(F.col('delay') == 0,  'No Delays')
     .otherwise('Early')
     .alias('Flight_Delays'))

df_labeled_sql.show(5)
df_labeled_api.show(5)

+-----+------+-----------+----------------+
|delay|origin|destination|   Flight_Delays|
+-----+------+-----------+----------------+
|    6|   ABE|        ATL|Tolerable Delays|
|   -8|   ABE|        DTW|           Early|
|   -2|   ABE|        ATL|           Early|
|   -4|   ABE|        ATL|           Early|
|   -4|   ABE|        ATL|           Early|
+-----+------+-----------+----------------+
only showing top 5 rows
+-----+------+-----------+----------------+
|delay|origin|destination|   Flight_Delays|
+-----+------+-----------+----------------+
|    6|   ABE|        ATL|Tolerable Delays|
|   -8|   ABE|        DTW|           Early|
|   -2|   ABE|        ATL|           Early|
|   -4|   ABE|        ATL|           Early|
|   -4|   ABE|        ATL|           Early|
+-----+------+-----------+----------------+
only showing top 5 rows


> You may also see an older nested style, `F.when(...).otherwise(F.when(...).otherwise(...))`. It gives the same result but is harder to read — chain `.when()` calls instead.

---
## 5. `GROUP BY` and `HAVING`
- `GROUP BY` splits the rows into groups and computes an aggregate (`COUNT`, `AVG`, `SUM`, `MIN`, `MAX`) per group.
- `WHERE` filters rows **before** grouping; `HAVING` filters **groups after** aggregation.

**Which origin airports have the highest average delay (only airports with more than 10,000 flights)?**

In [21]:
spark.sql("""
    SELECT origin,
           COUNT(*)             AS flights,
           ROUND(AVG(delay), 2) AS avg_delay
    FROM us_delay_flights_tbl
    GROUP BY origin
    HAVING COUNT(*) > 10000
    ORDER BY avg_delay DESC
""").show(10)

+------+-------+---------+
|origin|flights|avg_delay|
+------+-------+---------+
|   MDW|  20056|    19.66|
|   ORD|  64228|    18.59|
|   IAD|  14136|     18.4|
|   DEN|  53148|    16.92|
|   BWI|  21558|    16.83|
|   FLL|  18006|    16.51|
|   JFK|  23572|    16.46|
|   EWR|  27656|    16.37|
|   MCO|  28313|    15.72|
|   BNA|  13733|    15.45|
+------+-------+---------+
only showing top 10 rows


In [22]:
# DataFrame equivalent - HAVING is just a where() after agg()
(df.groupBy('origin')
   .agg(F.count('*').alias('flights'), F.round(F.avg('delay'), 2).alias('avg_delay'))
   .where('flights > 10000')
   .orderBy(F.desc('avg_delay'))
   .show(10))

+------+-------+---------+
|origin|flights|avg_delay|
+------+-------+---------+
|   MDW|  20056|    19.66|
|   ORD|  64228|    18.59|
|   IAD|  14136|     18.4|
|   DEN|  53148|    16.92|
|   BWI|  21558|    16.83|
|   FLL|  18006|    16.51|
|   JFK|  23572|    16.46|
|   EWR|  27656|    16.37|
|   MCO|  28313|    15.72|
|   BNA|  13733|    15.45|
+------+-------+---------+
only showing top 10 rows


In [23]:
# How many flights fall into each delay category? (uses the labelled DataFrame from section 4)
df_labeled_sql.createOrReplaceTempView("labeled_flights")
spark.sql("""
    SELECT Flight_Delays, COUNT(*) AS flights
    FROM labeled_flights
    GROUP BY Flight_Delays
    ORDER BY flights DESC
""").show()

+----------------+-------+
|   Flight_Delays|flights|
+----------------+-------+
|           Early| 668729|
|Tolerable Delays| 497742|
|       No Delays| 131122|
|    Short Delays|  61039|
|     Long Delays|  31447|
|Very Long Delays|   1499|
+----------------+-------+



---
## 6. Exercises (try them first, then look at the solutions)

### Exercise 6.1
Convert the `date` column into a readable format and find the **months** (or days) when the SFO → ORD delays were most common. Were they related to winter months or holidays?

*Hint:* `date` is `MMddHHmm`, so the first two characters are the month and the next two are the day.

In [24]:
# Solution 6.1 (SQL)
spark.sql("""
    SELECT CAST(substring(date, 1, 2) AS INT) AS month,
           CAST(substring(date, 3, 2) AS INT) AS day,
           COUNT(*)                           AS delayed_flights,
           MAX(delay)                         AS max_delay
    FROM us_delay_flights_tbl
    WHERE origin = 'SFO' AND destination = 'ORD' AND delay > 120
    GROUP BY month, day
    ORDER BY delayed_flights DESC, max_delay DESC
""").show(10)

+-----+---+---------------+---------+
|month|day|delayed_flights|max_delay|
+-----+---+---------------+---------+
|    1|  3|              4|      396|
|    1|  2|              4|      326|
|    3| 31|              4|      196|
|    3| 12|              3|      266|
|    2|  9|              3|      223|
|    1|  5|              2|      320|
|    3| 17|              2|      189|
|    3| 26|              2|      184|
|    2|  8|              2|      181|
|    1| 30|              2|      167|
+-----+---+---------------+---------+
only showing top 10 rows


In [25]:
# Solution 6.1 (DataFrame API) - build a real timestamp with to_timestamp.
# The data has no year, so Spark uses 1970; only month/day/time matter here.
sfo_ord = (df.where("origin = 'SFO' AND destination = 'ORD' AND delay > 120")
             .withColumn('departure', F.to_timestamp('date', 'MMddHHmm'))
             .withColumn('month', F.month('departure')))

sfo_ord.select('date', 'departure', 'delay').show(5, truncate=False)
sfo_ord.groupBy('month').count().orderBy('month').show()

+--------+-------------------+-----+
|date    |departure          |delay|
+--------+-------------------+-----+
|01011410|1970-01-01 14:10:00|124  |
|01022330|1970-01-02 23:30:00|326  |
|01021410|1970-01-02 14:10:00|190  |
|01101410|1970-01-10 14:10:00|184  |
|01190925|1970-01-19 09:25:00|297  |
+--------+-------------------+-----+
only showing top 5 rows


+-----+-----+
|month|count|
+-----+-----+
|    1|   22|
|    2|   17|
|    3|   16|
+-----+-----+



> The dataset only covers **January to March**, i.e. winter months — so the data cannot compare winter with summer. Look at which specific days have the most delays and whether they are close to holidays or known winter storms.

### Exercise 6.2
Convert the other two SQL queries from section 3 to the DataFrame API: *(a)* SFO → ORD with delay > 120, *(b)* the `CASE` labelling query. *(The `CASE` one is already done in 4.2.)*

In [26]:
# Solution 6.2 (a)
(df.select('date', 'delay', 'origin', 'destination')
   .where((F.col('delay') > 120) & (F.col('origin') == 'SFO') & (F.col('destination') == 'ORD'))
   .orderBy(F.desc('delay'))
   .show(10))

+--------+-----+------+-----------+
|    date|delay|origin|destination|
+--------+-----+------+-----------+
|02190925| 1638|   SFO|        ORD|
|01031755|  396|   SFO|        ORD|
|01022330|  326|   SFO|        ORD|
|01051205|  320|   SFO|        ORD|
|01190925|  297|   SFO|        ORD|
|02171115|  296|   SFO|        ORD|
|01071040|  279|   SFO|        ORD|
|01051550|  274|   SFO|        ORD|
|03120730|  266|   SFO|        ORD|
|01261104|  258|   SFO|        ORD|
+--------+-----+------+-----------+
only showing top 10 rows


---
## 7. Databases

### 7.1 Tables, metadata and the metastore
**Tables hold data.** Every table in Spark also has **metadata**: its schema, description, table name, database name, column names, partitions, and the **physical location** of the data.
All this metadata is stored in a central **metastore** (the *catalog*).

### 7.2 Creating and using a database
Tables live inside a **database**. By default Spark uses the database called `default`.

In [27]:
spark.sql("CREATE DATABASE IF NOT EXISTS learn_spark_db")   # IF NOT EXISTS: no error if it already exists
spark.sql("SHOW DATABASES").show()

Hive Session ID = c78d7359-d693-41f8-919b-dae60479d176


+--------------+
|     namespace|
+--------------+
|     astronomy|
|       default|
|      employee|
|learn_spark_db|
|      students|
+--------------+



In [28]:
spark.sql("USE learn_spark_db")      # from now on new tables are created in learn_spark_db
print(spark.catalog.currentDatabase())

learn_spark_db


In [29]:
# A database can also be stored in a folder you choose
spark.sql(f"CREATE DATABASE IF NOT EXISTS learn_spark_db_custom LOCATION '{OUT_DIR}/learn_spark_db_custom.db'")
spark.sql("DESCRIBE DATABASE learn_spark_db_custom").show(truncate=False)

+--------------+-----------------------------------------------------------------------+
|info_name     |info_value                                                             |
+--------------+-----------------------------------------------------------------------+
|Catalog Name  |spark_catalog                                                          |
|Namespace Name|learn_spark_db_custom                                                  |
|Comment       |                                                                       |
|Location      |hdfs://bdss/user/user/data/spark_course_output/learn_spark_db_custom.db|
|Owner         |user                                                                   |
+--------------+-----------------------------------------------------------------------+



### 7.3 Dropping a database
- `DROP DATABASE name` fails if the database still contains tables.
- `DROP DATABASE name CASCADE` also drops **all** its tables and functions.
- `IF EXISTS` avoids an error when the database does not exist.

In [30]:
spark.sql("DROP DATABASE IF EXISTS learn_spark_db_custom CASCADE")
spark.sql("SHOW DATABASES").show()

+--------------+
|     namespace|
+--------------+
|     astronomy|
|       default|
|      employee|
|learn_spark_db|
|      students|
+--------------+



---
## 8. Managed vs unmanaged tables

| | **Managed** table | **Unmanaged** (external) table |
|--|-------------------|-------------------------------|
| Spark manages the **metadata** | yes | yes |
| Spark manages the **data files** | yes — stored in the warehouse folder (local disk, HDFS, S3, Azure Blob, ...) | no — **you** own the files (your own path, or an external system such as Cassandra) |
| `DROP TABLE` deletes | the metadata **and the data** | **only the metadata** — the files stay |
| Created with | `CREATE TABLE ...` / `df.write.saveAsTable(...)` | `CREATE TABLE ... USING csv OPTIONS (path ...)` / `df.write.option("path", ...).saveAsTable(...)` |

Small helpers to see where a table's data lives, whether Spark manages it, and whether a folder exists in HDFS:

In [31]:
def table_info(table):
    """Show the Type (MANAGED / EXTERNAL), Provider (format), Location and Table Properties of a table."""
    (spark.sql(f"DESCRIBE TABLE EXTENDED {table}")
          .where("col_name IN ('Type', 'Provider', 'Location', 'Table Properties')")
          .show(truncate=False))

def table_path(table):
    """Folder (in HDFS) that holds the table's data files."""
    return (spark.sql(f"DESCRIBE TABLE EXTENDED {table}")
                 .where("col_name = 'Location'")
                 .collect()[0].data_type)

def hdfs_exists(path):
    """True if the path exists - uses Hadoop's FileSystem API through Spark's JVM."""
    jpath = spark._jvm.org.apache.hadoop.fs.Path(path)
    return jpath.getFileSystem(spark._jsc.hadoopConfiguration()).exists(jpath)

### 8.1 Managed tables
**(a) With SQL — create an empty table with a schema.**
`USING parquet` tells Spark which file format to store the table in.

> Without `USING ...`, Spark tries to create a **Hive** table, which only works when the session has Hive support (with Hive you see the warning *"A Hive serde table will be created as there is no table provider specified"*). Adding `USING parquet` works in both cases.

In [32]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS managed_us_delay_flights_tbl
        (date STRING, delay INT, distance INT, origin STRING, destination STRING)
    USING parquet
""")
spark.sql("SELECT * FROM managed_us_delay_flights_tbl").show()     # empty for now

+----+-----+--------+------+-----------+
|date|delay|distance|origin|destination|
+----+-----+--------+------+-----------+
+----+-----+--------+------+-----------+



In [33]:
# Fill it from our temporary view.
# INSERT INTO appends rows; INSERT OVERWRITE replaces them (so re-running this cell doesn't duplicate data).
spark.sql("INSERT OVERWRITE TABLE managed_us_delay_flights_tbl SELECT * FROM us_delay_flights_tbl")
spark.sql("SELECT COUNT(*) AS rows FROM managed_us_delay_flights_tbl").show()

+-------+
|   rows|
+-------+
|1391578|
+-------+



**(b) With SQL — `CREATE TABLE ... AS SELECT` (CTAS):** create and fill the table in one statement; the schema comes from the query.

In [34]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS managed_long_flights
    USING parquet
    AS SELECT * FROM us_delay_flights_tbl WHERE distance > 1000
""")
spark.sql("SELECT * FROM managed_long_flights").show(3)

+--------+-----+--------+------+-----------+
|    date|delay|distance|origin|destination|
+--------+-----+--------+------+-----------+
|01012355|    0|    1586|   ABQ|        JFK|
|01022355|  158|    1586|   ABQ|        JFK|
|01032355|    0|    1586|   ABQ|        JFK|
+--------+-----+--------+------+-----------+
only showing top 3 rows


**(c) With the DataFrame API — `saveAsTable`:**

In [35]:
df.write.saveAsTable("managed_us_delay_flights_tbl_2", mode='overwrite')
spark.sql("SELECT * FROM managed_us_delay_flights_tbl_2").show(3)

+--------+-----+--------+------+-----------+
|    date|delay|distance|origin|destination|
+--------+-----+--------+------+-----------+
|01011245|    6|     602|   ABE|        ATL|
|01020600|   -8|     369|   ABE|        DTW|
|01021245|   -2|     602|   ABE|        ATL|
+--------+-----+--------+------+-----------+
only showing top 3 rows


In [36]:
spark.sql("SHOW TABLES").show()     # isTemporary = true for our temp views
table_info("managed_us_delay_flights_tbl_2")

+--------------+--------------------+-----------+
|     namespace|           tableName|isTemporary|
+--------------+--------------------+-----------+
|learn_spark_db|managed_long_flights|      false|
|learn_spark_db|managed_us_delay_...|      false|
|learn_spark_db|managed_us_delay_...|      false|
|              |     labeled_flights|       true|
|              |us_delay_flights_tbl|       true|
+--------------+--------------------+-----------+

+----------------+-----------------------------------------------------------------------------------------------------+-------+
|col_name        |data_type                                                                                            |comment|
+----------------+-----------------------------------------------------------------------------------------------------+-------+
|Type            |EXTERNAL                                                                                             |       |
|Provider        |parquet        

**Surprise — the Type says `EXTERNAL`, not `MANAGED`!** This is a rule of the **Hive 3+/4 metastore**, not of Spark:
- In Hive 4, a "real" managed table must be a transactional (ACID) Hive table. Spark's Parquet tables are not ACID, so the metastore **translates** the managed table Spark asked for into an external one.
- Two table properties tell you it happened: **`TRANSLATED_TO_EXTERNAL=TRUE`** (Spark asked for managed) and **`external.table.purge=TRUE`** (dropping the table also deletes its files).
- The files go to Hive's external warehouse folder (`/user/hive/external/learn_spark_db.db/...`) instead of `/user/hive/warehouse`.

So in practice it **still behaves like a managed table**: Spark chose the location and `DROP TABLE` deletes the data (we prove it in 8.3).
Without Hive (or with an older metastore, as in the lecture), the same command shows `Type = MANAGED` with a location inside `spark.sql.warehouse.dir`.

### 8.2 Unmanaged tables
**(a) With SQL — a table on top of an existing CSV file.** Spark only records the metadata and reads the file when you query the table.

> **Important:** add `header 'true'` in `OPTIONS`. Without it the CSV header line is read as a data row (you get a row `date | null | null | origin | destination`).

In [37]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS us_delay_flights_csv_tbl
        (date STRING, delay INT, distance INT, origin STRING, destination STRING)
    USING csv
    OPTIONS (path '{flights_path}', header 'true')
""")
spark.sql("SELECT * FROM us_delay_flights_csv_tbl").show(3)
table_info("us_delay_flights_csv_tbl")

+--------+-----+--------+------+-----------+
|    date|delay|distance|origin|destination|
+--------+-----+--------+------+-----------+
|01011245|    6|     602|   ABE|        ATL|
|01020600|   -8|     369|   ABE|        DTW|
|01021245|   -2|     602|   ABE|        ATL|
+--------+-----+--------+------+-----------+
only showing top 3 rows
+----------------+-----------------------------------------------------------+-------+
|col_name        |data_type                                                  |comment|
+----------------+-----------------------------------------------------------+-------+
|Type            |EXTERNAL                                                   |       |
|Provider        |csv                                                        |       |
|Table Properties|[bucketing_version=2]                                      |       |
|Location        |hdfs://bdss/user/user/data/spark_course/departuredelays.csv|       |
+----------------+----------------------------------

**(b) With the DataFrame API — `option("path", ...)` + `saveAsTable`.** Spark writes the data to **your** folder (not the warehouse) and registers an external table on it.

> Always point `path` to a **new folder**. The path is where Spark *writes*; if you point it at your original data file you can destroy it.

In [38]:
unmanaged_dir = f"{OUT_DIR}/unmanaged_flights_data"

(df.write
   .option("path", unmanaged_dir)
   .mode("overwrite")
   .saveAsTable("us_delay_flights_unmanaged"))

spark.sql("SELECT * FROM us_delay_flights_unmanaged").show(3)
table_info("us_delay_flights_unmanaged")

+--------+-----+--------+------+-----------+
|    date|delay|distance|origin|destination|
+--------+-----+--------+------+-----------+
|01011245|    6|     602|   ABE|        ATL|
|01020600|   -8|     369|   ABE|        DTW|
|01021245|   -2|     602|   ABE|        ATL|
+--------+-----+--------+------+-----------+
only showing top 3 rows
+----------------+---------------------------------------------------------------------+-------+
|col_name        |data_type                                                            |comment|
+----------------+---------------------------------------------------------------------+-------+
|Type            |EXTERNAL                                                             |       |
|Provider        |parquet                                                              |       |
|Table Properties|[bucketing_version=2, numFilesErasureCoded=0]                        |       |
|Location        |hdfs://bdss/user/user/data/spark_course_output/unmanaged_flig

### 8.3 What does `DROP TABLE` really delete?

In [39]:
managed_dir = table_path("managed_us_delay_flights_tbl_2")
print("Before DROP -> managed files exist:", hdfs_exists(managed_dir),
      "| unmanaged files exist:", hdfs_exists(unmanaged_dir))

spark.sql("DROP TABLE managed_us_delay_flights_tbl_2")
spark.sql("DROP TABLE us_delay_flights_unmanaged")

print("After  DROP -> managed files exist:", hdfs_exists(managed_dir),
      "| unmanaged files exist:", hdfs_exists(unmanaged_dir))

Before DROP -> managed files exist: True | unmanaged files exist: True
After  DROP -> managed files exist: False | unmanaged files exist: True


- The **managed** table's folder is **gone** — Spark deleted metadata *and* data (on Hive 4 this is what `external.table.purge=TRUE` means).
- The **unmanaged** table's folder is **still there** — only the table name was removed. You can re-create a table on it or read it as files:

In [40]:
spark.read.parquet(unmanaged_dir).show(3)    # saveAsTable writes Parquet by default

+--------+-----+--------+------+-----------+
|    date|delay|distance|origin|destination|
+--------+-----+--------+------+-----------+
|01011245|    6|     602|   ABE|        ATL|
|01020600|   -8|     369|   ABE|        DTW|
|01021245|   -2|     602|   ABE|        ATL|
+--------+-----+--------+------+-----------+
only showing top 3 rows


---
## 9. Views
- A **view** is a saved query on top of existing tables (or DataFrames). You query it exactly like a table.
- Views **don't hold data**; tables do. Tables survive after the Spark application ends, **views disappear**.

| | Temporary view | Global temporary view |
|--|---------------|----------------------|
| Visible in | **only the SparkSession** that created it | **all SparkSessions** of the same Spark application |
| Lives in | the current session | the system database **`global_temp`** |
| Query it as | `view_name` | `global_temp.view_name` |
| SQL | `CREATE OR REPLACE TEMP VIEW` | `CREATE OR REPLACE GLOBAL TEMP VIEW` |
| DataFrame API | `df.createOrReplaceTempView()` | `df.createOrReplaceGlobalTempView()` |

**Example:** we only care about flights leaving **SFO** and **JFK**.

In [41]:
# Using SQL
spark.sql("""
    CREATE OR REPLACE GLOBAL TEMP VIEW us_origin_airport_SFO_global_tmp_view AS
    SELECT date, delay, origin, destination FROM us_delay_flights_tbl WHERE origin = 'SFO'
""")

spark.sql("""
    CREATE OR REPLACE TEMP VIEW us_origin_airport_JFK_tmp_view AS
    SELECT date, delay, origin, destination FROM us_delay_flights_tbl WHERE origin = 'JFK'
""")

DataFrame[]

In [42]:
# The same views using the DataFrame API
df_sfo = spark.sql("SELECT date, delay, origin, destination FROM us_delay_flights_tbl WHERE origin = 'SFO'")
df_jfk = df.select('date', 'delay', 'origin', 'destination').where("origin = 'JFK'")

df_sfo.createOrReplaceGlobalTempView("us_origin_airport_SFO_global_tmp_view")
df_jfk.createOrReplaceTempView("us_origin_airport_JFK_tmp_view")

In [43]:
# A global view needs the global_temp. prefix
spark.sql("SELECT * FROM global_temp.us_origin_airport_SFO_global_tmp_view").show(3)

# A normal temp view does not
spark.sql("SELECT * FROM us_origin_airport_JFK_tmp_view").show(3)

+--------+-----+------+-----------+
|    date|delay|origin|destination|
+--------+-----+------+-----------+
|01011250|   55|   SFO|        JFK|
|01012230|    0|   SFO|        JFK|
|01010705|   -7|   SFO|        JFK|
+--------+-----+------+-----------+
only showing top 3 rows
+--------+-----+------+-----------+
|    date|delay|origin|destination|
+--------+-----+------+-----------+
|01010900|   14|   JFK|        LAX|
|01011200|   -3|   JFK|        LAX|
|01011900|    2|   JFK|        LAX|
+--------+-----+------+-----------+
only showing top 3 rows


**See the scope difference yourself:** `spark.newSession()` creates a second session in the same application.

In [44]:
other_session = spark.newSession()

other_session.sql("SELECT COUNT(*) AS sfo_rows FROM global_temp.us_origin_airport_SFO_global_tmp_view").show()   # works

try:
    other_session.sql("SELECT * FROM us_origin_airport_JFK_tmp_view").show()
except Exception as e:
    print("Temp view is not visible in another session ->", type(e).__name__)

+--------+
|sfo_rows|
+--------+
|   39483|
+--------+



{"ts": "2026-10-30 05:28:08.433", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[TABLE_OR_VIEW_NOT_FOUND] The table or view `us_origin_airport_JFK_tmp_view` cannot be found. Verify the spelling and correctness of the schema and catalog.\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01", "context": {"errorClass": "TABLE_OR_VIEW_NOT_FOUND"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o448.sql.\n: org.apache.spark.sql.AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `us_origin_airport_JFK_tmp_view` cannot be found. Verify the spelling and correctness of the schema and catalog.\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error o

Temp view is not visible in another session -> AnalysisException


In [45]:
# Dropping views
spark.catalog.dropGlobalTempView("us_origin_airport_SFO_global_tmp_view")
spark.catalog.dropTempView("us_origin_airport_JFK_tmp_view")
# SQL alternative: DROP VIEW IF EXISTS global_temp.view_name  /  DROP VIEW IF EXISTS view_name

True

---
## 10. Reading tables and views into DataFrames
Any table or view can become a DataFrame:

In [46]:
flights_df1 = spark.sql("SELECT * FROM managed_us_delay_flights_tbl")
flights_df2 = spark.table("managed_us_delay_flights_tbl")
flights_df3 = spark.read.table("managed_us_delay_flights_tbl")     # same as spark.table

flights_df2.show(3)

+--------+-----+--------+------+-----------+
|    date|delay|distance|origin|destination|
+--------+-----+--------+------+-----------+
|01011245|    6|     602|   ABE|        ATL|
|01020600|   -8|     369|   ABE|        DTW|
|01021245|   -2|     602|   ABE|        ATL|
+--------+-----+--------+------+-----------+
only showing top 3 rows


In [47]:
# The catalog API lists what exists
for t in spark.catalog.listTables():
    print(f"{t.name:35} database={t.database}  type={t.tableType}  temporary={t.isTemporary}")

us_delay_flights_csv_tbl            database=learn_spark_db  type=EXTERNAL  temporary=False
managed_long_flights                database=learn_spark_db  type=EXTERNAL  temporary=False
managed_us_delay_flights_tbl        database=learn_spark_db  type=EXTERNAL  temporary=False
labeled_flights                     database=None  type=TEMPORARY  temporary=True
us_delay_flights_tbl                database=None  type=TEMPORARY  temporary=True


---
## 11. SQL directly on files
You don't have to load a file into a DataFrame first — SQL can create a view or an unmanaged table **directly** on top of files with `USING <format> OPTIONS (path ...)`.

### 11.1 Parquet

In [48]:
parquet_path = f"{DATA_DIR}/parquet/2010-summary.parquet"

spark.sql(f"""
    CREATE OR REPLACE TEMPORARY VIEW summary_parquet_view
    USING parquet
    OPTIONS (path '{parquet_path}')
""")
spark.sql("SELECT * FROM summary_parquet_view").show(5)

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Romania|    1|
|    United States|            Ireland|  264|
|    United States|              India|   69|
|            Egypt|      United States|   24|
|Equatorial Guinea|      United States|    1|
+-----------------+-------------------+-----+
only showing top 5 rows


In [49]:
# An unmanaged TABLE on the same files (survives in the catalog, DROP TABLE keeps the files)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS summary_parquet_tbl
        (DEST_COUNTRY_NAME STRING, ORIGIN_COUNTRY_NAME STRING, count LONG)
    USING parquet
    OPTIONS (path '{parquet_path}')
""")
spark.sql("SELECT * FROM summary_parquet_tbl ORDER BY count DESC").show(5)

+-----------------+-------------------+------+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME| count|
+-----------------+-------------------+------+
|    United States|      United States|348113|
|    United States|             Canada|  8305|
|           Canada|      United States|  8271|
|    United States|             Mexico|  6220|
|           Mexico|      United States|  6200|
+-----------------+-------------------+------+
only showing top 5 rows


In [50]:
# Shortcut: query a file without creating anything, with format.`path`
spark.sql(f"SELECT * FROM parquet.`{parquet_path}` WHERE DEST_COUNTRY_NAME = 'Egypt'").show()

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|            Egypt|      United States|   24|
+-----------------+-------------------+-----+



### 11.2 JSON

In [51]:
spark.sql(f"""
    CREATE OR REPLACE TEMPORARY VIEW summary_json_view
    USING json
    OPTIONS (path '{DATA_DIR}/json/*')
""")
spark.sql("SELECT * FROM summary_json_view").show(5)

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Romania|   15|
|    United States|            Croatia|    1|
|    United States|            Ireland|  344|
|            Egypt|      United States|   15|
|    United States|              India|   62|
+-----------------+-------------------+-----+
only showing top 5 rows


### 11.3 CSV
CSV options (`header`, `inferSchema`, `mode`, ...) go inside `OPTIONS`:

In [52]:
spark.sql(f"""
    CREATE OR REPLACE TEMPORARY VIEW summary_csv_view
    USING csv
    OPTIONS (path '{DATA_DIR}/csv/*', header 'true', inferSchema 'true', mode 'FAILFAST')
""")
spark.sql("""
    SELECT ORIGIN_COUNTRY_NAME, SUM(count) AS total_flights
    FROM summary_csv_view
    WHERE DEST_COUNTRY_NAME = 'United States'
    GROUP BY ORIGIN_COUNTRY_NAME
    ORDER BY total_flights DESC
""").show(10)

+-------------------+-------------+
|ORIGIN_COUNTRY_NAME|total_flights|
+-------------------+-------------+
|      United States|      2119795|
|             Canada|        49695|
|             Mexico|        38225|
|     United Kingdom|        10358|
|              Japan|         8643|
|            Germany|         8380|
| Dominican Republic|         7194|
|        The Bahamas|         5775|
|             France|         5290|
|           Colombia|         4981|
+-------------------+-------------+
only showing top 10 rows


---
## 12. User-Defined Functions in SQL
A Python function can be **registered** under a name so SQL queries can call it.
This lets, for example, a data scientist wrap an ML model in a UDF and a data analyst use it from plain SQL.

In [53]:
def convert_case(text):
    """'john jones' -> 'John Jones'"""
    if text is None:
        return None
    return " ".join(word[:1].upper() + word[1:] for word in text.split(" "))

spark.udf.register("convertUDFSQL", convert_case, StringType())

names_df = spark.createDataFrame([("1", "john jones"), ("2", "tracey smith"), ("3", "amy sanders")],
                                 ["Seqno", "Name"])
names_df.createOrReplaceTempView("NAME_TABLE")

spark.sql("SELECT Seqno, convertUDFSQL(Name) AS Name FROM NAME_TABLE").show(truncate=False)

+-----+------------+
|Seqno|Name        |
+-----+------------+
|1    |John Jones  |
|2    |Tracey Smith|
|3    |Amy Sanders |
+-----+------------+



> Python UDFs are slow (each row travels between the JVM and Python). Prefer built-in SQL functions when one exists — here `initcap(Name)` does the same job:

In [54]:
spark.sql("SELECT Seqno, initcap(Name) AS Name FROM NAME_TABLE").show(truncate=False)

+-----+------------+
|Seqno|Name        |
+-----+------------+
|1    |John Jones  |
|2    |Tracey Smith|
|3    |Amy Sanders |
+-----+------------+



---
## 13. SQL on streaming data
Structured Streaming is built on the Spark SQL engine, so a **streaming DataFrame can be registered as a temporary view** and queried with SQL. The result is again a streaming DataFrame.

*(See section 18 of the DataFrame notebook for the full streaming explanation.)*

**Before running:** from the Jupyter home page open **New → Terminal**, start a text server with `nc -lk 12345`, and type some words into it while the query runs.

In [55]:
spark.conf.set("spark.sql.shuffle.partitions", 4)   # small demo stream: 4 shuffle tasks per batch instead of 200

lines = (spark.readStream.format('socket')
         .option('host', 'localhost')
         .option('port', 12345)
         .load())

words  = lines.select(F.explode(F.split(F.col('value'), ' ')).alias('word'))
counts = words.groupBy('word').count()

counts.createOrReplaceTempView("wordcounttable")

# Sorting a stream is allowed only AFTER an aggregation and in complete output mode - both true here
top_words = spark.sql("SELECT * FROM wordcounttable ORDER BY count DESC")   # another streaming DataFrame
print(top_words.isStreaming)

True


In [56]:
query = (top_words.writeStream
         .format('console')
         .outputMode('complete')
         .trigger(processingTime='5 seconds')
         .option('checkpointLocation', f"{OUT_DIR}/chkpnt_sql_wordcount")
         .start())

query.awaitTermination(30)   # watch the batches for 30 seconds
query.stop()

-------------------------------------------
Batch: 0
-------------------------------------------
+----+-----+
|word|count|
+----+-----+
+----+-----+



-------------------------------------------
Batch: 1
-------------------------------------------
+-----+-----+
| word|count|
+-----+-----+
|   is|  212|
|spark|  212|
|hello|  106|
|world|  106|
| fast|  106|
|  fun|  106|
+-----+-----+



-------------------------------------------
Batch: 2
-------------------------------------------
+-----+-----+
| word|count|
+-----+-----+
|spark|  216|
|   is|  216|
|hello|  108|
|world|  108|
| fast|  108|
|  fun|  108|
+-----+-----+



-------------------------------------------
Batch: 3
-------------------------------------------
+-----+-----+
| word|count|
+-----+-----+
|spark|  226|
|   is|  226|
|hello|  113|
|world|  113|
| fast|  113|
|  fun|  113|
+-----+-----+



-------------------------------------------
Batch: 4
-------------------------------------------
+-----+-----+
| word|count|
+-----+-----+
|spark|  236|
|   is|  236|
|hello|  118|
|world|  118|
| fast|  118|
|  fun|  118|
+-----+-----+



-------------------------------------------
Batch: 5
-------------------------------------------
+-----+-----+
| word|count|
+-----+-----+
|spark|  246|
|   is|  246|
|hello|  123|
|world|  123|
| fast|  123|
|  fun|  123|
+-----+-----+



-------------------------------------------
Batch: 6
-------------------------------------------
+-----+-----+
| word|count|
+-----+-----+
|spark|  256|
|   is|  256|
|hello|  128|
|world|  128|
| fast|  128|
|  fun|  128|
+-----+-----+



---
## 14. Clean-up

In [57]:
spark.sql("USE default")
spark.sql("DROP DATABASE IF EXISTS learn_spark_db CASCADE")   # drops every table we created in it

# Folders that DROP does not remove:
#  - the unmanaged table's files survived DROP TABLE (that was the point of section 8.3)
#  - Hive 4 leaves the database's (now empty) folder in its external warehouse
for leftover in [unmanaged_dir, "/user/hive/external/learn_spark_db.db"]:
    jpath = spark._jvm.org.apache.hadoop.fs.Path(leftover)
    jpath.getFileSystem(spark._jsc.hadoopConfiguration()).delete(jpath, True)   # True = recursive

spark.catalog.clearCache()
spark.sql("SHOW DATABASES").show()

+---------+
|namespace|
+---------+
|astronomy|
|  default|
| employee|
| students|
+---------+



---
## 15. 📋 Cheat sheet — summary of everything

### The big ideas
| Idea | Remember |
|---|---|
| **Spark SQL** | The Spark module for structured data — SQL and the DataFrame API run on the **same engine** |
| `spark.sql("...")` | Always returns a **DataFrame** — lazy until an action like `.show()` |
| **Catalyst** | Analysis → logical optimization → physical planning → code generation. See it with `.explain(True)` |
| **Temporary view** | A **name** for a DataFrame so SQL can use it. Holds no data, disappears with the session |
| **Metastore / catalog** | Remembers databases, tables, schemas and locations. With `.enableHiveSupport()` it is the Hive metastore (survives restarts) |

### SQL ↔ DataFrame API

| Task | Spark SQL | DataFrame API |
|------|-----------|---------------|
| Make a DataFrame queryable | — | `df.createOrReplaceTempView("t")` |
| Choose columns | `SELECT a, b FROM t` | `df.select('a', 'b')` |
| Filter | `WHERE a > 5 AND b = 'x'` | `.where((F.col('a') > 5) & (F.col('b') == 'x'))` |
| Membership | `WHERE a IN ('x', 'y')` | `.where(F.col('a').isin('x', 'y'))` |
| Sort | `ORDER BY a DESC` | `.orderBy(F.desc('a'))` |
| If / else | `CASE WHEN ... THEN ... ELSE ... END` | `F.when(...).when(...).otherwise(...)` |
| Aggregate | `SELECT a, AVG(b) FROM t GROUP BY a` | `.groupBy('a').agg(F.avg('b'))` |
| Filter groups | `HAVING COUNT(*) > 10` | `.where(...)` after `.agg(...)` |
| Part of a string | `substring(date, 1, 2)` | `F.substring('date', 1, 2)` |
| Change type | `CAST(x AS INT)` | `F.col('x').cast('int')` |
| Create database | `CREATE DATABASE IF NOT EXISTS db` / `USE db` | `spark.catalog.setCurrentDatabase('db')` |
| Managed table | `CREATE TABLE t (...) USING parquet` / CTAS | `df.write.saveAsTable('t')` |
| Unmanaged table | `CREATE TABLE t (...) USING csv OPTIONS (path '...')` | `df.write.option('path', '...').saveAsTable('t')` |
| Temp view | `CREATE OR REPLACE TEMP VIEW v AS SELECT ...` | `df.createOrReplaceTempView('v')` |
| Global temp view | `CREATE OR REPLACE GLOBAL TEMP VIEW v AS ...` → `global_temp.v` | `df.createOrReplaceGlobalTempView('v')` |
| Table → DataFrame | `spark.sql("SELECT * FROM t")` | `spark.table('t')` |
| View on a file | `CREATE TEMP VIEW v USING parquet OPTIONS (path '...')` | `spark.read.parquet('...')` |
| Register UDF | `spark.udf.register('f', func, StringType())` | `F.udf(func, StringType())` |
| See the plan | `EXPLAIN SELECT ...` | `df.explain(True)` |

### Reading the CSV — the four variants
| Code | Result |
|---|---|
| `spark.read.csv(path)` | ❌ `_c0.._c4`, all strings, header line becomes a data row |
| `spark.read.csv(path, inferSchema=True, header=True)` | ⚠️ Correct, but reads the whole file once just to guess the types |
| `spark.read.csv(path, schema=s)` | ❌ Header line read as data → a junk row with nulls |
| `spark.read.csv(path, schema=s, header=True)` | ✅ Correct names and types, no inference job |

### Databases & the catalog
| Code | What it does |
|---|---|
| `CREATE DATABASE IF NOT EXISTS db` | Create (no error if it exists) |
| `CREATE DATABASE db LOCATION '/path'` | Store the database in a folder you choose |
| `USE db` / `spark.catalog.currentDatabase()` | Switch / show the current database |
| `SHOW DATABASES` / `SHOW TABLES` | List databases / tables and temp views |
| `DESCRIBE DATABASE db` / `DESCRIBE TABLE EXTENDED t` | Details: location, type, provider, properties |
| `DROP DATABASE db CASCADE` | Drop the database **and all its tables** |
| `spark.catalog.listTables()` | The same list from Python |

### Managed vs unmanaged tables
| | Managed | Unmanaged (external) |
|---|---|---|
| Who owns the data files? | Spark (warehouse folder) | **You** (your own path) |
| `DROP TABLE` deletes | metadata **and data** | **only metadata** — files stay |
| Create with SQL | `CREATE TABLE t (...) USING parquet` or `CREATE TABLE t USING parquet AS SELECT ...` | `CREATE TABLE t (...) USING csv OPTIONS (path '...', header 'true')` |
| Create with the API | `df.write.saveAsTable('t')` | `df.write.option('path', '...').saveAsTable('t')` |
| Fill / refill | `INSERT INTO t SELECT ...` (append) / `INSERT OVERWRITE TABLE t SELECT ...` (replace) | |

> On a **Hive 4** metastore a Spark "managed" table shows `Type = EXTERNAL` with `TRANSLATED_TO_EXTERNAL=TRUE` and `external.table.purge=TRUE` — it still behaves like a managed table.

### Views
| | Temporary view | Global temporary view |
|---|---|---|
| Visible in | only **this** SparkSession | **all** sessions of the application |
| Query as | `v` | `global_temp.v` |
| Drop | `spark.catalog.dropTempView('v')` | `spark.catalog.dropGlobalTempView('v')` |

### SQL directly on files & on streams
| Code | What it does |
|---|---|
| `CREATE OR REPLACE TEMPORARY VIEW v USING parquet OPTIONS (path '...')` | A view straight on Parquet files |
| `... USING json OPTIONS (path '.../json/*')` | Same for JSON (wildcards allowed) |
| `... USING csv OPTIONS (path '...', header 'true', inferSchema 'true')` | CSV options go inside `OPTIONS` |
| ``SELECT * FROM parquet.`/path/file.parquet` `` | Query a file without creating anything |
| `streaming_df.createOrReplaceTempView('t')` → `spark.sql(...)` | SQL on a stream returns another **streaming** DataFrame |

### ⚠️ Common mistakes to remember
| Mistake | Remember |
|---|---|
| `CASE` with `delay > 120 AND delay < 360` | Values **exactly** on the boundary fall into `ELSE` → order conditions largest-first and use one comparison each |
| `WHERE` vs `HAVING` | `WHERE` filters **rows before** grouping, `HAVING` filters **groups after** |
| CSV table without `header 'true'` | The header line becomes a data row |
| `CREATE TABLE` without `USING ...` | Tries to create a Hive serde table — add `USING parquet` |
| `option('path', ...)` pointing at your source data | `path` is where Spark **writes** — always use a new folder |
| Querying a global view without `global_temp.` | It is not found — global views live in the `global_temp` database |
| `DROP DATABASE db` that still has tables | Fails — add `CASCADE` |
| Python UDF when a built-in exists | Use the built-in (`initcap(Name)`) — UDFs are slow |
| `ORDER BY` on a stream | Only allowed after an aggregation and in `complete` output mode |
| 200 shuffle partitions on a tiny demo | `spark.conf.set("spark.sql.shuffle.partitions", 4)` |